# Environmental Solutions

:::{admonition} Reference solutions
:class: note
Worked solutions for [1.4-matplotlib-and-xarray-environmental-exercises.ipynb](1.4-matplotlib-and-xarray-environmental-exercises.ipynb). Run the setup cell below first. Every exercise has a worked solution, the four replication capstones included.
:::

In [ ]:
# Pre-supplied: run this first.
import numpy as np
import matplotlib.pyplot as plt

## Exercise 1: NASA's real global-temperature record

The file cached below is NASA GISS's monthly global-mean surface temperature anomaly (°C,
relative to a 1951–1980 baseline), one row per year since 1880, one column per calendar month.
A handful of recent months are still missing, marked `***`.

Wrap the twelve month columns in an xarray `DataArray` with dims `("year", "month")` and a `units`
attribute, then draw two figures: the annual mean as a line against year, and the full
`(year, month)` array with xarray's own `.plot()`. In a comment, compare what the warming trend
looks like read month by month against the single annual line.

In [ ]:
import xarray as xr
import pooch

path = pooch.retrieve(
    url="https://raw.githubusercontent.com/gse-unil/2026_MLEES_book/main/data/part-I/nasa_gistemp_global_temp_anomaly.csv",
    known_hash="sha256:6cfa44e7bbacd9b12cb10bdd64b3182c2735fa3f3a95688e1f7bc8e5dfcece93",
    fname="nasa_gistemp_global_temp_anomaly.csv",
    path=pooch.os_cache("mlees"),
)
data = np.genfromtxt(path, skip_header=1, delimiter=",", names=True,
                     missing_values="***", filling_values=np.nan)

anomaly = np.column_stack((data["Jan"], data["Feb"], data["Mar"], data["Apr"],
                           data["May"], data["Jun"], data["Jul"], data["Aug"],
                           data["Sep"], data["Oct"], data["Nov"], data["Dec"]))

da = xr.DataArray(
    anomaly,
    dims=("year", "month"),
    coords={"year": data["Year"], "month": np.arange(1, 13)},
    attrs={"units": "degC", "long_name": "global-mean surface temperature anomaly"},
)

fig, ax = plt.subplots(figsize=(7, 3))
da.mean(dim="month").plot(ax=ax, color="tab:red")   # skips missing months automatically
ax.set_xlabel("year")
ax.set_ylabel("temperature anomaly (°C)")
ax.set_title("annual-mean global temperature anomaly")
plt.show()

fig, ax = plt.subplots(figsize=(8, 4))
da.plot(ax=ax)          # colorbar labelled from long_name and units
ax.set_title("month-by-month anomaly")
plt.show()

# the annual line smooths out a within-year signal: the heatmap shows that recent decades
# are warm in every month, not just on average, and that the trend is not perfectly uniform
# across months

## Replicating plots

The four exercises below each show a figure built from a real dataset and ask you to rebuild it
as closely as you can. The data arrive through a pre-supplied `pooch` cell; everything after that
is yours.

Work towards the target one element at a time — get the data onto the axes first, then the
coordinates, then the colours, then the labels — rather than trying to write the whole cell in
one go.

## Exercise 2: Global surface air temperature and its zonal mean

The three files cached below hold one snapshot of surface air temperature from the
[NCEP/NCAR atmospheric reanalysis 1](https://psl.noaa.gov/data/gridded/data.ncep.reanalysis.html),
as plain numpy arrays: `lon` (192 longitudes), `lat` (94 latitudes, running north to south), and
`temp_kelvin`, the field itself with shape `(94, 192)` — **in kelvin**.

```{figure} _static/1.4-target-global-temperature.png
---
name: target-global-temperature
width: 100%
alt: A filled contour map of global surface air temperature in magma colours with a white dashed contour at minus ten degrees, beside a line plot of the zonal mean against latitude
---
The figure to replicate. Left: the field as filled contours, with a single white contour drawn
at −10 °C. Right: the zonal mean — the average along longitude — against latitude.
```

Rebuild it, and say in a comment which hemisphere's winter this snapshot was taken in, and how
the zonal mean tells you.

In [ ]:
# Pre-supplied: download the three arrays and cache them locally.
import pooch

base_url = "https://raw.githubusercontent.com/gse-unil/2026_MLEES_book/main/data/part-I/"

path_lon = pooch.retrieve(
    url=base_url + "ncep_global_temp_lon.npy",
    known_hash="sha256:eaf54b88dd89279d3034da17fe8470dc2c841bf9fa89b2aa741dacff9c326cdb",
    fname="ncep_global_temp_lon.npy",
    path=pooch.os_cache("mlees"),
)
path_lat = pooch.retrieve(
    url=base_url + "ncep_global_temp_lat.npy",
    known_hash="sha256:af1f438080460e1fca4583b2ec19b44285a3d3776e4d21b8da9b6e162906c88a",
    fname="ncep_global_temp_lat.npy",
    path=pooch.os_cache("mlees"),
)
path_temp = pooch.retrieve(
    url=base_url + "ncep_global_temp_kelvin.npy",
    known_hash="sha256:e040ca257334708b43e86398e09a5669fcf051179ecf5dcd278f758d67beed20",
    fname="ncep_global_temp_kelvin.npy",
    path=pooch.os_cache("mlees"),
)

In [ ]:
lon = np.load(path_lon)
lat = np.load(path_lat)
temp_celsius = np.load(path_temp) - 273.15      # the file is in kelvin
print(lon.shape, lat.shape, temp_celsius.shape)

fig, axes = plt.subplots(1, 2, figsize=(11, 5), gridspec_kw={"width_ratios": [5, 1.5]})
ax0, ax1 = axes

cf = ax0.contourf(lon, lat, temp_celsius, cmap="magma",
                  levels=np.linspace(-30, 40, 15), extend="both")
ax0.contour(lon, lat, temp_celsius, colors="w", levels=[-10], extend="both")
fig.colorbar(cf, ax=ax0, label="temperature (°C)")
ax0.set_xlabel("longitude (°E)")
ax0.set_ylabel("latitude (°N)")
ax0.set_title("global surface air temperature")

ax1.plot(np.nanmean(temp_celsius, axis=1), lat, linewidth=2, color="k")
ax1.set_xlabel("temperature (°C)")
ax1.set_ylabel("latitude (°N)")
ax1.set_title("zonal mean")
ax1.grid()

plt.tight_layout()
plt.show()

# northern-hemisphere winter: the zonal mean reaches about -39 °C at the northernmost latitude
# but only about -18 °C at the southernmost, and it stays near 26 °C out to 20° S while it has
# already dropped to about 22 °C by 20° N -- the warm belt is skewed towards the summer
# hemisphere, which here is the southern one

## Exercise 3: Historic significant earthquakes

The file cached below is NOAA's catalog of significant earthquakes, reaching back to 2150 BCE:
one row per event, tab-separated, 47 columns. The four you need are at positions 8
(`FOCAL_DEPTH`, km), 9 (`EQ_PRIMARY`, magnitude), 20 (`LATITUDE`) and 21 (`LONGITUDE`).

```{figure} _static/1.4-target-earthquakes-scatter.png
---
name: target-earthquakes-scatter
width: 100%
alt: A scatter plot of earthquake longitude against latitude, coloured by the base-10 logarithm of focal depth and sized by magnitude
---
The figure to replicate: every event as one point, coloured by the base-10 logarithm of its
focal depth and sized by its magnitude. No coastline is drawn — the plate boundaries are the
data.
```

Rebuild it, and in a comment name the feature of the Earth that the points trace out.

In [ ]:
import pooch

path = pooch.retrieve(
    url="https://raw.githubusercontent.com/gse-unil/2026_MLEES_book/main/data/part-I/signif.txt.tsv",
    known_hash="sha256:f27e00d335f64ffaf437e74a8efb54f9154d40b417ff93fefb0fd64856e6d345",
    fname="signif.txt.tsv",
    path=pooch.os_cache("mlees"),
)
arr = np.genfromtxt(path, delimiter="\t", skip_header=1)
focal_depth_km = arr[:, 8]
magnitude = arr[:, 9]
latitude = arr[:, 20]
longitude = arr[:, 21]

keep = (focal_depth_km > 0) & (magnitude > 0) & ~np.isnan(latitude) & ~np.isnan(longitude)
focal_depth_km, magnitude = focal_depth_km[keep], magnitude[keep]
latitude, longitude = latitude[keep], longitude[keep]
print("events kept:", keep.sum(), "of", keep.size)

fig, ax = plt.subplots(figsize=(10, 5))
sc = ax.scatter(longitude, latitude, s=magnitude ** 2, c=np.log10(focal_depth_km),
                marker="o", cmap="viridis", alpha=0.7)
fig.colorbar(sc, ax=ax, label="log10(focal depth in km)")
ax.set_xlabel("longitude (°E)")
ax.set_ylabel("latitude (°N)")
ax.set_title("historic significant earthquakes: depth and magnitude")
ax.grid(linestyle=":", linewidth=0.5)
plt.tight_layout()
plt.show()

# the points fall along plate boundaries, and the empty areas are plate interiors: the deepest
# events (yellow) sit where one plate is being subducted beneath another, along the Pacific rim
# and the Indonesian arc

## Exercise 4: Antarctic sea ice, summer against winter

Two real snapshots of Antarctic sea ice concentration from NOAA/NSIDC are cached below, as
`path_winter` (1 August 2017) and `path_summer` (31 December 2017). Each holds
`seaice_conc_cdr`, the ice concentration as a fraction from 0 to 1, on a south polar
stereographic grid — its coordinates are 2D `latitude`/`longitude` arrays rather than simple 1D
dimensions, so xarray's own `.plot()` will not label it cleanly.

```{figure} _static/1.4-target-seaice.png
---
name: target-seaice
width: 100%
alt: Two south polar stereographic maps of Antarctic sea ice concentration, one in austral winter with a wide ice ring and one in austral summer with much less ice
---
The figure to replicate: the same field on the same projection at two dates, with the land drawn
in grey beneath the ice and one colorbar shared by both panels.
```

Rebuild it, and say which date shows more ice.

In [ ]:
# Pre-supplied: download the data files and cache them locally.
import pooch

path_winter = pooch.retrieve(
    url="https://raw.githubusercontent.com/gse-unil/2026_MLEES_book/main/data/part-I/seaice_conc_daily_sh_f17_20170801_v03r01.nc",
    known_hash="sha256:1ff50bca1e6249a9b2fcd9d9466e31bdb5be650243f99c7319ab2ce625b87ce7",
    fname="seaice_conc_daily_sh_f17_20170801_v03r01.nc",
    path=pooch.os_cache("mlees"),
)
path_summer = pooch.retrieve(
    url="https://raw.githubusercontent.com/gse-unil/2026_MLEES_book/main/data/part-I/seaice_conc_daily_sh_f17_20171231_v03r01.nc",
    known_hash="sha256:309418969ad09f42b8104589bcb86de4ed353a5742fef9385baec174c7d55e66",
    fname="seaice_conc_daily_sh_f17_20171231_v03r01.nc",
    path=pooch.os_cache("mlees"),
)

In [ ]:
import xarray as xr
import cartopy.crs as ccrs
import cartopy.feature as cfeature

paths = [path_winter, path_summer]
titles = ["1 August 2017 (austral winter)", "31 December 2017 (austral summer)"]

fig, axes = plt.subplots(1, 2, figsize=(11, 5.5),
                         subplot_kw={"projection": ccrs.SouthPolarStereo()})

mean_conc = []
for ax, path, title in zip(axes, paths, titles):
    ds = xr.open_dataset(path)
    conc = ds["seaice_conc_cdr"].isel(time=0)
    conc = conc.where(conc <= 1)          # drop the land/coast/missing flag values above 1
    mean_conc.append(float(conc.mean()))

    ax.set_extent([-180, 180, -90, -55], ccrs.PlateCarree())
    ax.add_feature(cfeature.LAND, edgecolor="k", facecolor="0.8")
    ax.gridlines(linewidth=0.3, color="0.5")
    pcm = ax.pcolormesh(ds["longitude"], ds["latitude"], conc,
                        transform=ccrs.PlateCarree(), vmin=0, vmax=1, cmap="Blues_r")
    ax.set_title(title)
    ds.close()

fig.colorbar(pcm, ax=axes, label="sea ice concentration (fraction)")
plt.show()

print(f"mean concentration, winter: {mean_conc[0]:.3f} | summer: {mean_conc[1]:.3f}")

# August, the austral winter, shows far more ice: a broad ring reaching well away from the
# coast, against a thin and broken fringe at the end of December

## Exercise 5: The 2014 earthquakes over North America

The file cached below is the real USGS earthquake catalog for 2014 — one row per event
worldwide, with latitude, longitude, depth and magnitude in columns 1 to 4. The target figure
keeps the whole catalog and lets the map extent do the selecting, so what you see is a window
onto a global dataset rather than one cropped to North America in advance.

```{figure} _static/1.4-target-earthquakes-map.png
---
name: target-earthquakes-map
width: 90%
alt: A Robinson-projection map of North America with state boundaries, lakes and rivers, and earthquake locations coloured by depth
---
The figure to replicate: magnitude 4 and above, on a Robinson projection cropped to North and
Central America, over land, ocean, lake, river and state-boundary features.
```

Rebuild it.

In [ ]:
# Pre-supplied: download the data file and cache it locally.
import pooch

path = pooch.retrieve(
    url="https://raw.githubusercontent.com/gse-unil/2026_MLEES_book/main/data/part-I/usgs_earthquakes_2014.csv",
    known_hash="sha256:84d455fb96dc8f782fba4b5fbe56cb8970cab678f07c766fcba1b1c4674de1b1",
    fname="usgs_earthquakes_2014.csv",
    path=pooch.os_cache("mlees"),
)

In [ ]:
import cartopy.crs as ccrs
import cartopy.feature as cfeature

arr = np.genfromtxt(path, delimiter=",", skip_header=1, usecols=(1, 2, 3, 4))
latitude, longitude, depth_km, mag = arr[:, 0], arr[:, 1], arr[:, 2], arr[:, 3]

keep = (mag >= 4) & (depth_km > 0)
latitude, longitude = latitude[keep], longitude[keep]
depth_km, mag = depth_km[keep], mag[keep]
print("events kept:", keep.sum(), "of", keep.size)

fig, ax = plt.subplots(figsize=(9, 7), subplot_kw={"projection": ccrs.Robinson()})
ax.set_extent([-140, -60, 12, 70], ccrs.PlateCarree())

ax.add_feature(cfeature.LAND, edgecolor="k", linewidth=1.0, facecolor="0.92")
ax.add_feature(cfeature.OCEAN)
ax.add_feature(cfeature.LAKES)
ax.add_feature(cfeature.RIVERS)
ax.add_feature(cfeature.STATES, edgecolor="gray", linewidth=0.6)
ax.gridlines(linewidth=0.3, color="0.5")

sc = ax.scatter(longitude, latitude, transform=ccrs.PlateCarree(),
                c=np.log10(depth_km), s=mag**2, cmap="viridis", alpha=0.7)
fig.colorbar(sc, ax=ax, label="log10(depth in km)")
ax.set_title("2014 earthquakes, magnitude 4 and above")
plt.show()

# the map window is what selects: the catalog is worldwide, and what shows up here is the
# Aleutian arc, the Californian plate boundary, the Central American trench, and a tight
# cluster in Oklahoma that is induced rather than tectonic